# GeoLift-RT v2.1 — Train/Val/Test one-run từ hai TAR 2.000

Notebook này **không tải/copy teacher mới và không tạo lại TAR**. Nó chỉ dùng ba file đã chuẩn bị sẵn trên My Drive:

```text
MyDrive/GeoLift_Data/teacher_subset_2000/
├── metric_coarse_train_2000.tar
├── geometry_fused_train_2000.tar
└── selected_2000_ids.json
```

Split cố định:

```text
1.600 train
400 validation
1.000 KITTI anonymous test
```

Teacher được dùng:

- `metric_coarse`: `D_cm, C_cm`
- `geometry_fused`: `R_G, C_G`

`depth_anything_train_raw_private.tar` không được load và không tham gia loss.

Flow:

```text
mount Drive
→ copy repo sang SSD local
→ verify hai TAR + manifest 1.600/400
→ extract 2.000 NPZ/role xuống /content
→ lấy đúng KITTI RGB/sparse/GT/calibration
→ coverage/schema gate
→ loader visualization
→ resolve config + smoke test
→ train/resume
→ validation + KITTI test + runtime profile
```


In [ ]:
from pathlib import Path
from getpass import getpass
import subprocess
import os
import stat

# ============================================================
# CONFIG
# ============================================================

REPO_DIR = Path(
    "/content/drive/MyDrive/DEPTH-FUSION | Workspace/"
    "monocular_sparse_fusion/GeoRT"
)

GITHUB_USERNAME = "PhuocDang2104"
GITHUB_REPO_URL = "https://github.com/PhuocDang2104/GeoDistill_RT.git"

GIT_EMAIL = input("Nhập email GitHub của bạn: ").strip()
COMMIT_MESSAGE = input(
    "Commit message [Update GeoDistill-RT]: "
).strip() or "Update GeoDistill-RT"

GITHUB_TOKEN = getpass("Nhập GitHub Personal Access Token: ").strip()

assert REPO_DIR.is_dir(), f"Không tìm thấy repo: {REPO_DIR}"
assert (REPO_DIR / ".git").is_dir(), f"Thư mục chưa phải Git repo: {REPO_DIR}"
assert GIT_EMAIL, "Email không được để trống."
assert GITHUB_TOKEN, "GitHub token không được để trống."

def run_git(*args, check=True, env=None, capture_output=False):
    command = ["git", *args]
    print("\n$", " ".join(command))

    return subprocess.run(
        command,
        cwd=REPO_DIR,
        check=check,
        text=True,
        env=env,
        capture_output=capture_output,
    )

# ============================================================
# GIT CONFIG
# ============================================================

run_git("config", "user.name", GITHUB_USERNAME)
run_git("config", "user.email", GIT_EMAIL)

# Chuẩn hóa branch thành main
run_git("branch", "-M", "main")

# Thêm hoặc cập nhật remote origin
remote_result = run_git(
    "remote",
    capture_output=True,
)

remotes = remote_result.stdout.split()

if "origin" in remotes:
    run_git("remote", "set-url", "origin", GITHUB_REPO_URL)
else:
    run_git("remote", "add", "origin", GITHUB_REPO_URL)

run_git("remote", "-v")

# ============================================================
# ADD + COMMIT
# ============================================================

run_git("add", ".")

has_staged_changes = subprocess.run(
    ["git", "diff", "--cached", "--quiet"],
    cwd=REPO_DIR,
).returncode != 0

if has_staged_changes:
    run_git("commit", "-m", COMMIT_MESSAGE)
else:
    print("\nKhông có thay đổi mới để commit. Tiếp tục push commit hiện tại.")

# ============================================================
# AUTHENTICATION KHÔNG HIỂN THỊ TOKEN
# ============================================================

askpass_file = Path("/content/github_askpass.sh")

askpass_file.write_text(
    """#!/bin/sh
case "$1" in
    *Username*) echo "$GITHUB_USERNAME" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
""",
    encoding="utf-8",
)

askpass_file.chmod(
    stat.S_IRUSR |
    stat.S_IWUSR |
    stat.S_IXUSR
)

git_env = os.environ.copy()
git_env["GIT_ASKPASS"] = str(askpass_file)
git_env["GIT_TERMINAL_PROMPT"] = "0"
git_env["GITHUB_USERNAME"] = GITHUB_USERNAME
git_env["GITHUB_TOKEN"] = GITHUB_TOKEN

# ============================================================
# PUSH
# ============================================================

try:
    run_git(
        "push",
        "-u",
        "origin",
        "main",
        env=git_env,
    )

    print("\n✅ Push thành công:")
    print("https://github.com/PhuocDang2104/GeoDistill_RT")

finally:
    askpass_file.unlink(missing_ok=True)
    GITHUB_TOKEN = ""
    git_env.pop("GITHUB_TOKEN", None)

Nhập email GitHub của bạn: phuoc.dang2104@gmail.com
Commit message [Update GeoDistill-RT]: updated


In [ ]:
# 0) Mount Drive và cấu hình toàn bộ run
from google.colab import drive
from pathlib import Path
import os
import sys
import io
import re
import json
import time
import shutil
import tarfile
import random
import hashlib
import subprocess

import numpy as np
import torch

drive.mount("/content/drive")

# ============================================================
# REPO
# ============================================================

DRIVE_REPO_DIR = Path(
    "/content/drive/MyDrive/DEPTH-FUSION | Workspace/"
    "monocular_sparse_fusion/GeoRT"
)

LOCAL_REPO = Path("/content/GeoDistill_RT")

# ============================================================
# HAI TAR FINAL 2.000 + MANIFEST SPLIT 1.600/400
# ============================================================

DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/GeoLift_Data")
TAR2000_ROOT = DRIVE_DATA_ROOT / "teacher_subset_2000"

METRIC_TAR = TAR2000_ROOT / "metric_coarse_train_2000.tar"
GEOMETRY_TAR = TAR2000_ROOT / "geometry_fused_train_2000.tar"
FINAL_MANIFEST = TAR2000_ROOT / "selected_2000_ids.json"

# ============================================================
# LOCAL SSD
# ============================================================

DATA_ROOT = Path("/content/geolift_data")
DOWNLOADS = DATA_ROOT / "downloads"
TEACHER_ROOT = DATA_ROOT / "teacher_outputs"
SPLIT_ROOT = DATA_ROOT / "splits"
INSPECT_ROOT = DATA_ROOT / "inspect"
KITTI_ROOT = DATA_ROOT / "kitti"

RUN_LOCAL = Path("/content/geolift_final_run")
RUN_DRIVE = Path(
    "/content/drive/MyDrive/GeoLift_RT_runs/"
    "v2_1_metric_geometry_train1600_val400"
)

# ============================================================
# RUN SETTINGS
# ============================================================

FINAL_COUNT = 2000
TRAIN_COUNT = 1600
VAL_COUNT = 400
TEST_COUNT = 1000

SEED = 2026
EPOCHS = 30
BATCH_SIZE = 2
NUM_WORKERS = 2

assert TRAIN_COUNT + VAL_COUNT == FINAL_COUNT

for path in (
    DATA_ROOT,
    DOWNLOADS,
    TEACHER_ROOT,
    SPLIT_ROOT,
    INSPECT_ROOT,
    KITTI_ROOT,
    RUN_LOCAL,
    RUN_DRIVE,
):
    path.mkdir(parents=True, exist_ok=True)

assert DRIVE_REPO_DIR.is_dir(), f"Không tìm thấy repo: {DRIVE_REPO_DIR}"
assert METRIC_TAR.is_file(), f"Thiếu TAR metric 2000: {METRIC_TAR}"
assert GEOMETRY_TAR.is_file(), f"Thiếu TAR geometry 2000: {GEOMETRY_TAR}"
assert FINAL_MANIFEST.is_file(), f"Thiếu manifest 2000: {FINAL_MANIFEST}"
assert torch.cuda.is_available(), "Hãy chọn Colab GPU runtime trước khi Run all."

print("GPU:", torch.cuda.get_device_name(0))
print("Metric TAR:", METRIC_TAR)
print("Geometry TAR:", GEOMETRY_TAR)
print("Manifest:", FINAL_MANIFEST)
print("Run backup:", RUN_DRIVE)


Mounted at /content/drive
GPU: Tesla T4
Metric TAR: /content/drive/MyDrive/GeoLift_Data/teacher_subset_2000/metric_coarse_train_2000.tar
Geometry TAR: /content/drive/MyDrive/GeoLift_Data/teacher_subset_2000/geometry_fused_train_2000.tar
Manifest: /content/drive/MyDrive/GeoLift_Data/teacher_subset_2000/selected_2000_ids.json
Run backup: /content/drive/MyDrive/GeoLift_RT_runs/v2_1_metric_geometry_train1600_val400


## 1. Copy source code từ Drive sang SSD local

Chỉ copy source cần thiết; bỏ `.git`, dữ liệu lớn, cache và output cũ.


In [ ]:
# 1) Copy repo sang /content để train/infer nhanh hơn Google Drive mount
assert DRIVE_REPO_DIR.is_dir(), DRIVE_REPO_DIR

if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)

shutil.copytree(
    DRIVE_REPO_DIR,
    LOCAL_REPO,
    ignore=shutil.ignore_patterns(
        ".git",
        "venv",
        ".venv",
        "data",
        "teacher_outputs",
        "student_outputs",
        "runs",
        "__pycache__",
        "*.pyc",
    ),
)

os.chdir(LOCAL_REPO)

for required in ("src", "scripts", "configs", "tests"):
    path = LOCAL_REPO / required
    assert path.exists(), f"Repo thiếu: {path}"

requirements = LOCAL_REPO / "requirements.txt"
if requirements.is_file():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )

print("Local repo:", LOCAL_REPO)
print("Python:", sys.version)


Local repo: /content/GeoDistill_RT
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


## 2. Verify hai TAR 2.000 và manifest 1.600/400

Notebook dừng ngay nếu:

- Hai TAR không có cùng canonical ID.
- Manifest không có đúng `train_ids` và `val_ids`.
- Train/val trùng sample hoặc trùng raw drive.


In [ ]:
# 2) Strict preflight TAR + manifest
def canonical_id(member_name: str) -> str:
    stem = Path(member_name).stem

    old_format = re.match(
        r"^(?P<prefix>.+)_sync_image_"
        r"(?P<camera>\d{2})_(?P<frame>\d{10})$",
        stem,
    )
    if old_format:
        return (
            f"{old_format.group('prefix')}_sync_image_"
            f"{old_format.group('frame')}_image_"
            f"{old_format.group('camera')}"
        )

    canonical_format = re.match(
        r"^(?P<prefix>.+)_sync_image_"
        r"(?P<frame>\d{10})_image_(?P<camera>\d{2})$",
        stem,
    )
    if canonical_format:
        return stem

    raise ValueError(f"Không canonicalize được: {member_name}")


def raw_drive(sample_id: str) -> str:
    return sample_id.split("_sync_image_")[0]


def index_tar(tar_path: Path) -> dict:
    result = {}

    print(f"Indexing {tar_path.name}...")

    with tarfile.open(tar_path, "r:*") as tar:
        for member in tar.getmembers():
            if not member.isfile() or not member.name.endswith(".npz"):
                continue

            sample_id = canonical_id(member.name)

            if sample_id in result:
                raise RuntimeError(
                    f"Duplicate canonical ID trong {tar_path.name}: {sample_id}"
                )

            result[sample_id] = member.name

    print(f"  NPZ: {len(result):,}")
    return result


final_manifest = json.loads(FINAL_MANIFEST.read_text())

assert "train_ids" in final_manifest, (
    "Manifest không có train_ids. Cần dùng selected_2000_ids.json "
    "đã được tạo theo split 1.600/400."
)
assert "val_ids" in final_manifest, (
    "Manifest không có val_ids. Cần dùng selected_2000_ids.json "
    "đã được tạo theo split 1.600/400."
)

train_ids = sorted(final_manifest["train_ids"])
val_ids = sorted(final_manifest["val_ids"])
selected_ids = sorted(train_ids + val_ids)

assert len(train_ids) == TRAIN_COUNT, len(train_ids)
assert len(val_ids) == VAL_COUNT, len(val_ids)
assert len(selected_ids) == FINAL_COUNT, len(selected_ids)
assert len(set(selected_ids)) == FINAL_COUNT
assert not (set(train_ids) & set(val_ids))

train_drives = sorted({raw_drive(item) for item in train_ids})
val_drives = sorted({raw_drive(item) for item in val_ids})
assert not (set(train_drives) & set(val_drives)), (
    "Train/val đang trùng raw drive."
)

metric_index = index_tar(METRIC_TAR)
geometry_index = index_tar(GEOMETRY_TAR)

metric_ids = set(metric_index)
geometry_ids = set(geometry_index)
manifest_ids = set(selected_ids)

assert len(metric_ids) == FINAL_COUNT
assert len(geometry_ids) == FINAL_COUNT
assert metric_ids == geometry_ids == manifest_ids

print("\n===== TAR 2000 PREFLIGHT PASSED =====")
print("Train:", len(train_ids), "| raw drives:", len(train_drives))
print("Val  :", len(val_ids), "| raw drives:", len(val_drives))
print("Drive overlap:", 0)
print("Metric size  :", f"{METRIC_TAR.stat().st_size / 1024**3:.2f} GiB")
print("Geometry size:", f"{GEOMETRY_TAR.stat().st_size / 1024**3:.2f} GiB")


AssertionError: Manifest không có train_ids. Cần dùng selected_2000_ids.json đã được tạo theo split 1.600/400.

## 3. Extract toàn bộ 2.000 teacher sample xuống SSD

Hai role được extract theo đúng cùng `selected_ids`. Cache local cũ sẽ bị xóa để tránh DA hoặc sample thừa.


In [ ]:
# 3) Extract 2.000 NPZ/role và tạo manifest local cho KITTI extractor
def extract_selected_npz(
    tar_path: Path,
    tar_index: dict,
    output_dir: Path,
    ids: list,
    role: str,
):
    output_dir.mkdir(parents=True, exist_ok=True)

    with tarfile.open(tar_path, "r:*") as source_tar:
        member_map = {
            member.name: member
            for member in source_tar.getmembers()
        }

        selected_members = []
        for sample_id in ids:
            member = member_map[tar_index[sample_id]]
            selected_members.append(
                (member.offset_data, sample_id, member)
            )

        # Đọc theo offset để giảm random seek trên Drive mount.
        selected_members.sort(key=lambda item: item[0])

        for number, (_, sample_id, member) in enumerate(
            selected_members,
            start=1,
        ):
            file_obj = source_tar.extractfile(member)
            assert file_obj is not None, member.name

            output_path = output_dir / f"{sample_id}.npz"
            with output_path.open("wb") as destination:
                shutil.copyfileobj(file_obj, destination)

            if number % 100 == 0 or number == len(ids):
                print(f"{role}: {number:,}/{len(ids):,}")


# Luôn tạo cache sạch cho một run mới.
if TEACHER_ROOT.exists():
    shutil.rmtree(TEACHER_ROOT)

metric_dir = TEACHER_ROOT / "metric_coarse" / "train"
geometry_dir = TEACHER_ROOT / "geometry_fused" / "train"

extract_selected_npz(
    METRIC_TAR,
    metric_index,
    metric_dir,
    selected_ids,
    role="Metric extract",
)

extract_selected_npz(
    GEOMETRY_TAR,
    geometry_index,
    geometry_dir,
    selected_ids,
    role="Geometry extract",
)

assert len(list(metric_dir.glob("*.npz"))) == FINAL_COUNT
assert len(list(geometry_dir.glob("*.npz"))) == FINAL_COUNT

subset_manifest = SPLIT_ROOT / "teacher_subset_manifest.json"

manifest_data = {
    "version": 4,
    "roles": ["metric_coarse", "geometry_fused"],
    "count": FINAL_COUNT,
    "train_count": TRAIN_COUNT,
    "val_count": VAL_COUNT,
    "seed": final_manifest.get("seed", SEED),
    "selection_strategy": "fixed_tar2000_raw_drive_disjoint",
    "train_ids": train_ids,
    "val_ids": val_ids,
    "selected_ids": selected_ids,
    "train_drives": train_drives,
    "val_drives": val_drives,
    "drives": {
        "train": train_drives,
        "val": val_drives,
    },
    "id_overlap_train_val": 0,
    "drive_overlap_train_val": 0,
    "teacher_root": str(TEACHER_ROOT),
}

subset_manifest.write_text(
    json.dumps(manifest_data, indent=2)
)

print("Teacher root:", TEACHER_ROOT)
print("Subset manifest:", subset_manifest)
print("Free local SSD:", f"{shutil.disk_usage('/content').free / 1024**3:.2f} GiB")


## 4. Lấy đúng KITTI frame và tạo split path

Tải KITTI official ZIP, chỉ giữ các frame tương ứng với 1.600 train, 400 val và 1.000 anonymous test.


In [ ]:
# 4) KITTI train/val subset + official anonymous test 1000
KITTI_URL = "https://s3.eu-central-1.amazonaws.com/avg-kitti"

official = {
    "annotated": (
        "data_depth_annotated.zip",
        f"{KITTI_URL}/data_depth_annotated.zip",
    ),
    "velodyne": (
        "data_depth_velodyne.zip",
        f"{KITTI_URL}/data_depth_velodyne.zip",
    ),
    "selection": (
        "data_depth_selection.zip",
        f"{KITTI_URL}/data_depth_selection.zip",
    ),
}

expected_splits = {
    "train_1600.txt": TRAIN_COUNT,
    "val_400.txt": VAL_COUNT,
    "test_1000.txt": TEST_COUNT,
}


def split_is_ready() -> bool:
    for filename, expected_count in expected_splits.items():
        split_path = SPLIT_ROOT / filename

        if not split_path.is_file():
            return False

        actual_count = sum(
            bool(line.strip())
            for line in split_path.read_text().splitlines()
        )

        if actual_count != expected_count:
            return False

    return True


if split_is_ready():
    print("KITTI subset đã tồn tại đúng count; bỏ qua download/extract.")
else:
    for filename, url in official.values():
        output_path = DOWNLOADS / filename

        subprocess.run(
            [
                "wget",
                "-q",
                "--show-progress",
                "-c",
                url,
                "-O",
                str(output_path),
            ],
            check=True,
        )

    subprocess.run(
        [
            sys.executable,
            "scripts/extract_official_kitti_subset.py",
            "--manifest",
            str(subset_manifest),
            "--annotated_zip",
            str(DOWNLOADS / official["annotated"][0]),
            "--velodyne_zip",
            str(DOWNLOADS / official["velodyne"][0]),
            "--selection_zip",
            str(DOWNLOADS / official["selection"][0]),
            "--data_root",
            str(KITTI_ROOT),
            "--split_root",
            str(SPLIT_ROOT),
            "--download_dir",
            str(DOWNLOADS),
        ],
        cwd=LOCAL_REPO,
        check=True,
    )

    # ZIP chỉ là file trung gian.
    for filename, _ in official.values():
        (DOWNLOADS / filename).unlink(missing_ok=True)

assert split_is_ready(), (
    "KITTI extractor chưa tạo đúng train_1600/val_400/test_1000."
)

print("KITTI splits OK:", expected_splits)


## 5. Gate B — coverage và schema hai teacher role

Kiểm tra 100% coverage và inspect định lượng một tập mẫu train/val trước khi model được khởi tạo.


In [ ]:
# 5) Coverage/schema/statistics gate cho metric + fused geometry
subset_report = INSPECT_ROOT / "teacher_train_val_report.json"
all_ids = train_ids + val_ids

role_dirs = {
    "metric_coarse": metric_dir,
    "geometry_fused": geometry_dir,
}

coverage = {}

for role, role_dir in role_dirs.items():
    present = {path.stem for path in role_dir.glob("*.npz")}

    missing = sorted(set(all_ids) - present)
    extras = sorted(present - set(all_ids))

    coverage[role] = {
        "expected": len(all_ids),
        "present": len(present),
        "missing": len(missing),
        "extras": len(extras),
        "ratio": len(set(all_ids) & present) / len(all_ids),
    }

    assert not missing, f"{role} thiếu ID: {missing[:10]}"
    assert not extras, f"{role} có sample thừa: {extras[:10]}"


stats = {
    "metric_D_cm_min": float("inf"),
    "metric_D_cm_max": float("-inf"),
    "metric_C_cm_min": float("inf"),
    "metric_C_cm_max": float("-inf"),
    "geometry_R_G_min": float("inf"),
    "geometry_R_G_max": float("-inf"),
    "geometry_C_G_min": float("inf"),
    "geometry_C_G_max": float("-inf"),
}

inspection_ids = (
    random.Random(SEED).sample(train_ids, min(16, len(train_ids)))
    + random.Random(SEED + 1).sample(val_ids, min(16, len(val_ids)))
)

for sample_id in inspection_ids:
    metric_path = metric_dir / f"{sample_id}.npz"
    geometry_path = geometry_dir / f"{sample_id}.npz"

    with np.load(metric_path, allow_pickle=False) as npz:
        assert {"D_cm", "C_cm"} <= set(npz.files)

        D_cm = np.asarray(npz["D_cm"])
        C_cm = np.asarray(npz["C_cm"])

        assert D_cm.shape == C_cm.shape
        assert np.isfinite(D_cm).all()
        assert np.isfinite(C_cm).all()
        assert float(C_cm.min()) >= 0.0
        assert float(C_cm.max()) <= 1.0

        stats["metric_D_cm_min"] = min(
            stats["metric_D_cm_min"], float(D_cm.min())
        )
        stats["metric_D_cm_max"] = max(
            stats["metric_D_cm_max"], float(D_cm.max())
        )
        stats["metric_C_cm_min"] = min(
            stats["metric_C_cm_min"], float(C_cm.min())
        )
        stats["metric_C_cm_max"] = max(
            stats["metric_C_cm_max"], float(C_cm.max())
        )

    with np.load(geometry_path, allow_pickle=False) as npz:
        assert {"R_G", "C_G"} <= set(npz.files)

        R_G = np.asarray(npz["R_G"])
        C_G = np.asarray(npz["C_G"])

        assert R_G.shape == C_G.shape
        assert np.isfinite(R_G).all()
        assert np.isfinite(C_G).all()
        assert float(C_G.min()) >= 0.0
        assert float(C_G.max()) <= 1.0

        stats["geometry_R_G_min"] = min(
            stats["geometry_R_G_min"], float(R_G.min())
        )
        stats["geometry_R_G_max"] = max(
            stats["geometry_R_G_max"], float(R_G.max())
        )
        stats["geometry_C_G_min"] = min(
            stats["geometry_C_G_min"], float(C_G.min())
        )
        stats["geometry_C_G_max"] = max(
            stats["geometry_C_G_max"], float(C_G.max())
        )


report = {
    "contract_ok": True,
    "roles": ["metric_coarse", "geometry_fused"],
    "train_count": len(train_ids),
    "val_count": len(val_ids),
    "id_overlap": 0,
    "drive_overlap": 0,
    "coverage": coverage,
    "inspected_samples": len(inspection_ids),
    "stats": stats,
}

subset_report.write_text(json.dumps(report, indent=2))

assert all(
    role_report["ratio"] == 1.0
    for role_report in coverage.values()
)

print(json.dumps(report, indent=2))


## 6. Loader thật và visualization

Load đúng pipeline train/val với `load_mono=False`; không có `D_da_raw`.


In [ ]:
# 6) Dataset loader thật + visualization train/val
import matplotlib.pyplot as plt
from src.dataset import KITTIDepthCompletionDataset


def inspect_split(split_name: str, split_file: str):
    dataset = KITTIDepthCompletionDataset(
        data_root=KITTI_ROOT,
        split_root=SPLIT_ROOT,
        split_file=split_file,
        split_name=split_name,
        image_size=(352, 1216),
        teacher_root=TEACHER_ROOT,
        load_teacher=True,
        load_geometry=True,
        geometry_fallback=False,
        load_mono=False,
        return_tensors=True,
    )

    sample = dataset[0]

    assert float(sample["C_G"].sum()) > 0
    assert torch.isfinite(sample["R_G"]).all()
    assert torch.isfinite(sample["D_cm"]).all()
    assert "D_da_raw" not in sample

    panels = [
        ("RGB", sample["rgb"].permute(1, 2, 0).numpy(), None),
        ("Sparse", sample["sparse"][0].numpy(), "turbo"),
        ("GT", sample["gt"][0].numpy(), "turbo"),
        ("Metric D_cm", sample["D_cm"][0].numpy(), "turbo"),
        ("Fused R_G", sample["R_G"][0].numpy(), "turbo"),
        ("Fused C_G", sample["C_G"][0].numpy(), "viridis"),
    ]

    figure, axes = plt.subplots(2, 3, figsize=(20, 8))

    for axis, (title, image, cmap) in zip(axes.ravel(), panels):
        axis.imshow(image, cmap=cmap)
        axis.set_title(title)
        axis.axis("off")

    figure.suptitle(f"{split_name}: {sample['sample_id']}")
    plt.tight_layout()
    plt.show()

    print(
        split_name,
        {
            key: tuple(sample[key].shape)
            for key in (
                "rgb",
                "sparse",
                "gt",
                "D_cm",
                "C_cm",
                "R_G",
                "C_G",
            )
        },
    )

    return dataset


train_dataset = inspect_split("train", "train_1600.txt")
val_dataset = inspect_split("val", "val_400.txt")

assert len(train_dataset) == TRAIN_COUNT
assert len(val_dataset) == VAL_COUNT


## 7. Resolve config final

Bật metric + fused geometry, tắt DA raw, tắt mono SSI fallback và tự resume từ `last.pth` trên Drive nếu có.


In [ ]:
# 7) Resolve config cho đúng hai teacher role
import yaml

paths_file = DATA_ROOT / "paths_geolift_tar2000.yaml"

paths = {
    "data_root": str(KITTI_ROOT),
    "split_root": str(SPLIT_ROOT),
    "train_split": "train_1600.txt",
    "val_split": "val_400.txt",
    "test_split": "test_1000.txt",
    "teacher_root": str(TEACHER_ROOT),
    "student_root": str(RUN_LOCAL),
}

paths_file.write_text(
    yaml.safe_dump(paths, sort_keys=False)
)

base_config_path = LOCAL_REPO / "configs" / "geolift_s2_v2_1.yaml"
assert base_config_path.is_file(), base_config_path

cfg = yaml.safe_load(base_config_path.read_text())

cfg["paths_file"] = str(paths_file)
cfg["train"]["epochs"] = EPOCHS
cfg["train"]["batch_size"] = BATCH_SIZE
cfg["data"]["num_workers"] = NUM_WORKERS
cfg["data"]["load_mono"] = False
cfg["outputs"]["backup_root"] = str(RUN_DRIVE)

cfg.setdefault("loss", {})
cfg["loss"]["geometry_fallback"] = False

cfg.setdefault("teacher_checks", {})
cfg["teacher_checks"]["require_geometry_fused"] = True
cfg["teacher_checks"]["require_da_raw"] = False
cfg["teacher_checks"]["require_mono_raw"] = False

cfg.setdefault("mono_ssi", {})
cfg["mono_ssi"]["enabled"] = False

# Resume tự động từ checkpoint đã backup trên Drive.
last_checkpoint = RUN_DRIVE / "checkpoints" / "last.pth"

if last_checkpoint.exists():
    local_checkpoint_dir = RUN_LOCAL / "checkpoints"
    local_checkpoint_dir.mkdir(parents=True, exist_ok=True)

    local_last = local_checkpoint_dir / "last.pth"
    shutil.copy2(last_checkpoint, local_last)
    cfg["train"]["resume"] = str(local_last)

resolved_cfg = DATA_ROOT / "geolift_s2_tar2000_train1600_val400.yaml"
resolved_cfg.write_text(
    yaml.safe_dump(cfg, sort_keys=False)
)

artifacts_to_copy = [
    (resolved_cfg, "resolved_config.yaml"),
    (paths_file, "resolved_paths.yaml"),
    (FINAL_MANIFEST, "selected_2000_ids.json"),
    (subset_manifest, "teacher_subset_manifest.json"),
    (subset_report, "teacher_train_val_report.json"),
]

for source, name in artifacts_to_copy:
    shutil.copy2(source, RUN_LOCAL / name)

run_manifest = {
    "architecture": "GeoLift-S2-v2.1",
    "teacher_profile": "metric D_cm/C_cm + fused R_G/C_G",
    "source_profile": "prebuilt matched TAR 2000",
    "da_raw_used": False,
    "geometry_fallback": False,
    "slope_supervision": False,
    "train": TRAIN_COUNT,
    "val": VAL_COUNT,
    "test_anonymous": TEST_COUNT,
    "seed": final_manifest.get("seed", SEED),
    "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "config_sha256": hashlib.sha256(
        resolved_cfg.read_bytes()
    ).hexdigest(),
}

(RUN_LOCAL / "run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2)
)

print(resolved_cfg.read_text())


## 8. Unit tests, strict preflight và one-batch smoke

Chưa chạy train nếu unit test, teacher coverage, geometry loss hoặc sparse identity bị lỗi.


In [ ]:
# 8) Tests + real loader/model/loss smoke
subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        "tests",
        "-v",
    ],
    cwd=LOCAL_REPO,
    check=True,
)

from src.utils import load_project_config
from src.train_student import make_loader, preflight_teacher_coverage
from src.model_factory import build_student
from src.losses import geort_loss
import logging

loaded_cfg, loaded_paths = load_project_config(resolved_cfg)

train_loader = make_loader(
    loaded_cfg,
    loaded_paths,
    "train",
    training=True,
)

preflight_teacher_coverage(
    loaded_cfg,
    train_loader.dataset,
    logging.getLogger("strict-preflight"),
)

batch = next(iter(train_loader))

assert "D_cm" in batch
assert "R_G" in batch
assert "C_G" in batch
assert "D_da_raw" not in batch

model = build_student(loaded_cfg).cuda().train()

gpu_batch = {
    key: (
        value[:1].cuda()
        if torch.is_tensor(value)
        else value
    )
    for key, value in batch.items()
}

with torch.autocast("cuda", dtype=torch.float16):
    prediction = model(
        *(
            gpu_batch[key]
            for key in (
                "rgb",
                "sparse",
                "mask",
                "ray",
                "uv",
                "K",
            )
        )
    )

    loss, items = geort_loss(
        prediction,
        gpu_batch,
        loaded_cfg["loss"],
        loaded_cfg["schedule"],
        loaded_cfg["schedule"]["add_geometry_epoch"],
        loaded_cfg["mono_ssi"],
    )

assert torch.isfinite(loss)
assert items["geom_valid_ratio"] > 0

sparse_identity_error = (
    (
        prediction["D_full"]
        - gpu_batch["sparse"]
    )
    * gpu_batch["mask"]
).abs().max()

assert float(sparse_identity_error) == 0.0

print("Strict smoke loss:", float(loss.detach()))
print("Geometry valid ratio:", items["geom_valid_ratio"])
print("Sparse identity max error:", float(sparse_identity_error))

del model, prediction, batch, gpu_batch
torch.cuda.empty_cache()


## 9. Train hoặc resume

Repo backup checkpoint/log sang `RUN_DRIVE`; cuối cell đồng bộ lại toàn bộ artifacts.


In [ ]:
# 9) Final train/resume
subprocess.run(
    [
        sys.executable,
        "-m",
        "src.train_student",
        "--config",
        str(resolved_cfg),
    ],
    cwd=LOCAL_REPO,
    check=True,
)

subprocess.run(
    [
        "rsync",
        "-a",
        f"{RUN_LOCAL}/",
        f"{RUN_DRIVE}/",
    ],
    check=True,
)

best_checkpoint = RUN_LOCAL / "checkpoints" / "best.pth"
assert best_checkpoint.exists(), best_checkpoint

print("Training complete:", best_checkpoint)


## 10. Validation, KITTI anonymous test và component runtime

Xuất:

- Global validation metrics.
- 1.000 benchmark PNG.
- ZIP nộp KITTI.
- Component runtime profile.


In [ ]:
# 10) Validation + anonymous test + runtime profile
for split_name in ("val", "test"):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "src.infer_student",
            "--config",
            str(resolved_cfg),
            "--checkpoint",
            str(best_checkpoint),
            "--split",
            split_name,
        ],
        cwd=LOCAL_REPO,
        check=True,
    )

metric_file = RUN_LOCAL / "logs" / "infer_val_metrics_global.json"
assert metric_file.is_file(), metric_file
print(metric_file.read_text())

test_prediction_dir = (
    RUN_LOCAL
    / "test_predictions"
    / "benchmark_png"
)

test_png = list(test_prediction_dir.glob("*.png"))
assert len(test_png) == TEST_COUNT, len(test_png)

shutil.make_archive(
    str(RUN_LOCAL / "kitti_test_1000"),
    "zip",
    test_prediction_dir,
)

profile_file = (
    RUN_LOCAL
    / "logs"
    / "geolift_component_profile.json"
)

subprocess.run(
    [
        sys.executable,
        "scripts/profile_geolift_s2.py",
        "--config",
        str(resolved_cfg),
        "--warmup",
        "30",
        "--runs",
        "100",
        "--output",
        str(profile_file),
    ],
    cwd=LOCAL_REPO,
    check=True,
)

subprocess.run(
    [
        "rsync",
        "-a",
        f"{RUN_LOCAL}/",
        f"{RUN_DRIVE}/",
    ],
    check=True,
)

print("Validation metrics:", metric_file)
print("KITTI ZIP:", RUN_LOCAL / "kitti_test_1000.zip")
print("Runtime profile:", profile_file)
print("Saved final run to:", RUN_DRIVE)


## Output cuối

```text
MyDrive/GeoLift_RT_runs/v2_1_metric_geometry_train1600_val400/
├── checkpoints/
│   ├── best.pth
│   └── last.pth
├── logs/
│   ├── infer_val_metrics_global.json
│   └── geolift_component_profile.json
├── selected_2000_ids.json
├── teacher_subset_manifest.json
├── teacher_train_val_report.json
├── resolved_config.yaml
├── resolved_paths.yaml
├── run_manifest.json
└── kitti_test_1000.zip
```

Notebook không copy, xóa hoặc sửa các TAR trên My Drive. Hai TAR 2.000 luôn được giữ nguyên để có thể re-run sau khi Colab reset.
